# Profiles

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pyprojroot import here
from ruamel.yaml import YAML


In [ ]:
# Get project root
root_path = Path(here())
yaml=YAML()
with open(root_path.joinpath('notebooks/settings.yml'), "r") as f:
    cfg = yaml.load(f)

# info of diagnostic tool and generated webpages
work_root = cfg['workdir']['work_root']
www_root = cfg['jupyterbook']['www_root']

# info of experiments
caseid = cfg['experiment']['caseid']
input_root = cfg['experiment']['input_root']
grid_label = cfg['experiment']['grid_label']
year_start = cfg['experiment']['year_start']
year_end = cfg['experiment']['year_end']

# info of grids
grid_file = cfg['grids'][grid_label]['grid_file']
region_file = cfg["grids"][grid_label]["region_file"]
grid_file = cfg['grids'][grid_label]['grid_file']
mask_RECCAP2_blom_file = cfg['grids'][grid_label]['mask_RECCAP2_file']

# info of observational datasets
phc_file = cfg['datasets']['PHC3.0']['ANNUAL_file']
phc_file = cfg['datasets']['PHC3.0']['ANNUAL_file']
mask_RECCAP2_file = cfg['datasets']['PHC3.0']['mask_RECCAP2_file']

# info of model climatologies
climo_file = cfg['climos']['blom']['hy']['ANN']


In [ ]:
# mapping of variable names from PHC to BLOM
varmappings = {'temp': 'templvl', 'salt': 'salnlvl'}

with xr.open_dataset(grid_file) as f:
    areacello = f['parea']

with xr.open_dataset(mask_RECCAP2_blom_file) as f:
    mask_blom = f['arctic']
    region_names = f['region_names']

with xr.open_dataset(mask_RECCAP2_file) as f:
    mask_phc = f['arctic']

# option 1: selective regions to compute and plot
# basin_names = ['Central Basin (CB)']
# region_ids = [region_names.region.values[region_names.values == basin_name][0] for basin_name in basin_names]
# option 2: all regions
region_ids = np.arange(1,region_names.region.size+1)

for region_id in region_ids:
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 8))
    k = 0   # order of plots in the subplots
    for varphc, varblom in varmappings.items():
        # read BLOM data and compute profile
        with xr.open_dataset(climo_file) as f:
            da = f[varblom].isel(time=0)
            profile = da.where(mask_blom == region_id).weighted(areacello).mean(dim=['y','x'])
            profile_index = np.arange(profile.depth.size)

        # read PHC data and compute profile
        with xr.open_dataset(phc_file) as f:
            da_phc = f[varphc]
            profile_phc = da_phc.where(mask_phc == region_id).weighted(np.cos(np.deg2rad(da_phc["lat"]))).mean(dim=['lat','lon'])

        # interpolate PHC to model depth so that both profiles can be plotted with non-linear depth
        profile_phc_on_blom_depth = profile_phc.interp(depth=profile.depth, method='linear')

        # plot the profiles
        axs[k].plot(profile, profile_index, color='r', label='BLOM')
        axs[k].plot(profile_phc_on_blom_depth, profile_index, color='k', label='PHC3.0')
        axs[k].set_xlabel(f"{da.long_name} ({da.units})")
        axs[k].set_ylabel("Depth [m]")

        # move ytick and ylabel to the right
        if k == 1:
            axs[k].yaxis.tick_right()
            axs[k].yaxis.set_label_position("right")

        axs[k].set_yticks(profile_index[::5])
        axs[k].set_yticklabels(profile['depth'].values[::5])
        axs[k].set_title(f'Profile of {da.long_name} in {region_names[region_id-1].values}')
        axs[k].invert_yaxis()
        axs[k].grid(linestyle='--', alpha=0.5)
        axs[k].legend()
        k = k + 1

    plt.show()
